In [ ]:
import os
import yaml

# --- 1. INSTALLAZIONE LIBRERIA UFFICIALE YOLO ---
# Ultralytics supporta nativamente YOLOv9 (modelli yolov9c o yolov9e)
os.system("pip install ultralytics")

In [ ]:
from ultralytics import YOLO

# --- 2. AGGIORNAMENTO PERCORSI YAML ---

# !!! ATTENZIONE !!!
# modificare i path
# Il percorso in input dove si trova la cartella 'images' e 'labels' dello split2
percorso_base = "/inserisci/il/path/alla/cartella/split2"

# Yaml per il Training
yaml_train = {
    "path": percorso_base, 
    "train": "images/train", 
    "val": "images/val",
    "nc": 2, 
    "names": ["Volto", "Targa"]
}

# Yaml per Test Normali (Colonna Sinistra Tabella)
yaml_test_normal = {
    "path": percorso_base, 
    "train": "images/train", 
    "val": "images/test_normal",
    "nc": 2, 
    "names": ["Volto", "Targa"]
}

# Yaml per Test Fisheye (Colonna Destra Tabella)
yaml_test_fisheye = {
    "path": percorso_base, 
    "train": "images/train", 
    "val": "images/test_fisheye",
    "nc": 2, 
    "names": ["Volto", "Targa"]
}

# Salvataggio dei nuovi file yaml in working
with open("/inserisci/il/path/di/output/per/dataset_train.yaml", "w") as f: yaml.dump(yaml_train, f, sort_keys = False)
with open("/inserisci/il/path/di/output/per/dataset_test_normal.yaml", "w") as f: yaml.dump(yaml_test_normal, f, sort_keys = False)
with open("/inserisci/il/path/di/output/per/dataset_test_fisheye.yaml", "w") as f: yaml.dump(yaml_test_fisheye, f, sort_keys = False)

print("Ambiente pronto e file YAML aggiornati correttamente!")

In [ ]:
# --- 1. INIZIALIZZAZIONE DEL MODELLO ---
# Scarica l'architettura YOLOv9 compatta (bilanciamento perfetto precisione/velocità)
print("Caricamento architettura YOLOv9c...")
modello = YOLO("yolov9c.pt")

# --- 2. AVVIO DELL'ADDESTRAMENTO BLINDATO ---
print("Inizio addestramento su PP4AV con salvataggi intermedi...")

risultati = modello.train(
    data = "/inserisci/il/path/per/dataset_train.yaml",                   # Path al file YAML che abbiamo generato prima
    seed = 42,                                                            # Seme per la riproducibilità
    epochs = 50,                                                          # Numero di epoche (consigliato 50 per iniziare)
    imgsz = 640,                                                          # Dimensione input con letterboxing automatico
    batch = 16,                                                           # Dimensione del batch (se ti dà Out of Memory, scendi a 8)
    project = "/inserisci/il/path/di/output/per/training_pp4av_split2",   # Cartella principale sicura per i salvataggi
    name = "run_yolov9",                                                  # Sottocartella di questo specifico esperimento
    device = 0,                                                           # Forza l'uso della GPU
    workers = 4,                                                          # Numero di thread per caricare i dati velocemente
    pretrained = True,                                                    # Usa i pesi COCO come base di partenza (Transfer Learning)
    save_period = 5,                                                      # Salva un checkpoint ogni 5 epoche
    cache = True                                                          # Metti "True" solo se hai tanta RAM, altrimenti lascia False
)

print("Addestramento completato con successo! Modello salvato in /kaggle/working/training_pp4av_split2/run_yolov9/weights/best.pt")